Quickly download and start using a pre-trained model to generate motion, then view it instantly with our BVH viewer.

Start by importing the necessary libraries and functions from the project, and set the device

In [ ]:
import torch
from torch.amp import autocast
import time
import matplotlib.pyplot as plt
from model import ContinuousMotionModel
from dataset.dataset import *
import utils.utils as utils
import soundfile as sf
from IPython.display import clear_output
from utils.performance_tracker import PerformanceTracker

device = utils.get_device()

Then, load the model that you downloaded from the provided link in the README. The models should be placed in the trained models folder, from where it can be loaded by the following cells.

In [ ]:
model_path = "models/normal/normal_seed_2__3aq6g23e_epoch_901.pth"

# Load the model
model: ContinuousMotionModel = ContinuousMotionModel.load_model(model_path, device)
model.condition_mask_probabilty = 0.0  # Disable condition mask probability for inference
model = model.to(device)
model.eval() # Set the model to evaluation mode

To see if it worked, print the number of params in the model:

In [ ]:
num_params = sum(p.numel() for p in model.parameters())
print(f"Number of parameters in the model: {num_params}")

Now start the bvh viewer. Open the link and run cell below it to start the generation of poses. These should be rendered at streamed to the viewer. 

In [ ]:
import utils.animation.visualisation.new.animation_visualisation as animation_visualisation
print(animation_visualisation.init_visualization(display=False))

In [ ]:
# Sliding Diffusion inference loop
with autocast(device_type=device.type, dtype=torch.bfloat16):
    dataset = GPUDataset(
        consolidated_file="dataset/genea2023_dataset/val/main-agent/consolidated.npz",
        seq_length=70,
        seed_length=8,
        batch_size=1,
        epoch_length=1,  # Set to 1 for testing purposes
        return_audio_frame_index=True,  # Set to True to return the audio frame index
    )

    # Use no gradient calculation for inference
    with torch.no_grad():

        gesture_sequence, seed_gesture, _, main_agent_id_one_hot, start_frames = [
            item.to(device) for item in next(iter(dataset))
        ]
        full_audio_features = dataset.audio.to(device)
        start_frame = start_frames[0].item()  # Extract the first element from the tensor

        # Decode the input using the autoencoder model
        if model.pose_encoder is not None:
            encoded_gesture_sequence = model.pose_encoder.encode(gesture_sequence)
        else:
            encoded_gesture_sequence = gesture_sequence

        iteration_counter = 0
        
        while True:
            iteration_counter += 1
            # Start time for the current frame
            frame_start_time = time.time()

            # The audio features also have to be shifted by one frame
            # I have the full audio features and the starting frame, so I extract the audio features for the current frame
            actual_audio_features = full_audio_features[start_frame + iteration_counter: start_frame + iteration_counter + dataset.seq_length, :].unsqueeze(0)

            encoded_gesture_sequence, noisy_gesture_sequence = model.inference(encoded_gesture_sequence, actual_audio_features, main_agent_id_one_hot, seed_gesture)

            ########################################################################################################################################################################

            # Decode the output using the autoencoder model
            clear_output(wait=True)

            # In case the model added extra features for richer embeddings, we only take the original features for decoding
            denoised_frame_to_decode = encoded_gesture_sequence[:,model.diffusion.clean_frame_index,:model.original_pose_features_per_frame].unsqueeze(0)
            
            pre_decode_time = time.time()
            if model.pose_encoder is not None:
                unencoded_denoised_frame = model.pose_encoder.decode(denoised_frame_to_decode)
            else:
                unencoded_denoised_frame = denoised_frame_to_decode
            post_decode_time = time.time()
            
            print(f"Time taken for decoding in ms: {(post_decode_time - pre_decode_time) * 1000:.2f} ms")

            denmormalized_unencoded_denoised_frame = dataset.skeleton.denormalize_poses(unencoded_denoised_frame).squeeze(0).squeeze(0)
            # print(f"Shape of denormalized unencoded denoised frame: {denmormalized_unencoded_denoised_frame.shape}")
            animation_visualisation.send_pose(denmormalized_unencoded_denoised_frame.cpu(), dataset.skeleton)
            animation_visualisation.send_debug_tensor(torch.cat((actual_audio_features.squeeze(0).to(torch.float32),noisy_gesture_sequence.squeeze(0).to(torch.float32)), dim=1), "full tensor")


            print(f"Iteration: {iteration_counter}")
            frame_end_time = time.time()

            frame_time = frame_end_time - frame_start_time
            sliding_tracker.record_frame(frame_time)

            # Print the time taken for the current frame
            print(f"Frame {iteration_counter} processed in {frame_time:.4f} seconds ({1/(frame_end_time - frame_start_time):.2f} FPS)")

            # Sleep for the remaining time in the 30 FPS frame
            time_to_sleep = max(0, (1/30) - frame_time - 0.0005)  # 0.01 is a small buffer to account for processing time
            time.sleep(time_to_sleep)